## 1. Import libraries and load the cleaned dataset

In [1]:
import pandas as pd
import sqlite3
from pathlib import Path

DATA_DIR = Path("../dataset")
REPORT_DIR = Path("../reports")
SQL_DIR = Path("../sql")

REPORT_DIR.mkdir(parents=True, exist_ok=True)
SQL_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_DIR / "steam_games_cleaned.csv")
print("Rows:", len(df))
print("Columns:", df.columns.tolist())
display(df.head())


Rows: 491
Columns: ['name', 'release_date', 'estimated_owners', 'price', 'positive', 'negative', 'average_playtime_forever', 'genres']


,name,release_date,estimated_owners,price,positive,negative,average_playtime_forever,genres
0,Shadow Empire Deluxe 2049,2013-05-17,"0 .. 20,000",5.99,629.0,163.0,171.0,"Indie, Horror"
1,Crimson Rift Origins,2013-06-12,"100,000 .. 200,000",0.99,6293.0,23.0,NaN,Free To Play
2,Forgotten Legends,2012-04-14,"10,000,000 .. 20,000,000",14.99,690.0,76.0,152.0,Strategy
3,Last Empire Origins,2012-03-25,"20,000 .. 50,000",4.99,19009.0,486.0,43.0,"Puzzle, Sports"
4,Iron Tactics Deluxe 2,2009-05-24,"20,000 .. 50,000",59.99,3820.0,163.0,181.0,"Sports, Puzzle"


## 2. Create SQLite database and table `steam_games`

In [24]:
db_path = DATA_DIR / "steam_insight.db"
conn = sqlite3.connect(db_path)

df.to_sql("steam_games", conn, if_exists="replace", index=False)

print("Database created:", db_path)
display(pd.read_sql_query("SELECT * FROM steam_games LIMIT 5;", conn))


Database created: ..\dataset\steam_insight.db


,name,release_date,estimated_owners,price,positive,negative,average_playtime_forever,genres
0,Shadow Empire Deluxe 2049,2013-05-17,"0 .. 20,000",5.99,629.0,163.0,171.0,"Indie, Horror"
1,Crimson Rift Origins,2013-06-12,"100,000 .. 200,000",0.99,6293.0,23.0,NaN,Free To Play
2,Forgotten Legends,2012-04-14,"10,000,000 .. 20,000,000",14.99,690.0,76.0,152.0,Strategy
3,Last Empire Origins,2012-03-25,"20,000 .. 50,000",4.99,19009.0,486.0,43.0,"Puzzle, Sports"
4,Iron Tactics Deluxe 2,2009-05-24,"20,000 .. 50,000",59.99,3820.0,163.0,181.0,"Sports, Puzzle"


## 3. Verify table structure

In [25]:
display(pd.read_sql_query("PRAGMA table_info(steam_games);", conn))
display(pd.read_sql_query("SELECT COUNT(*) AS total_rows FROM steam_games;", conn))


,cid,name,type,notnull,dflt_value,pk
0,0,name,TEXT,0,None,0
1,1,release_date,TEXT,0,None,0
2,2,estimated_owners,TEXT,0,None,0
3,3,price,REAL,0,None,0
4,4,positive,REAL,0,None,0
5,5,negative,REAL,0,None,0
6,6,average_playtime_forever,REAL,0,None,0
7,7,genres,TEXT,0,None,0


,total_rows
0,491


## 4. SQL business questions

### Q1 — First 10 games

In [26]:
q1 = '''\nSELECT *
FROM steam_games
LIMIT 10;\n'''

result = pd.read_sql_query(q1, conn)
display(result)

,name,release_date,estimated_owners,price,positive,negative,average_playtime_forever,genres
0,Shadow Empire Deluxe 2049,2013-05-17,"0 .. 20,000",5.99,629.0,163.0,171.0,"Indie, Horror"
1,Crimson Rift Origins,2013-06-12,"100,000 .. 200,000",0.99,6293.0,23.0,NaN,Free To Play
2,Forgotten Legends,2012-04-14,"10,000,000 .. 20,000,000",14.99,690.0,76.0,152.0,Strategy
3,Last Empire Origins,2012-03-25,"20,000 .. 50,000",4.99,19009.0,486.0,43.0,"Puzzle, Sports"
4,Iron Tactics Deluxe 2,2009-05-24,"20,000 .. 50,000",59.99,3820.0,163.0,181.0,"Sports, Puzzle"
5,Iron Survival,2007-08-24,"0 .. 20,000",9.99,289.0,116.0,85.0,"Puzzle, Simulation, Strategy, Platformer"
6,Cyber Horizon 2049,2018-05-27,"200,000 .. 500,000",19.99,785.0,46.0,38.0,Strategy
7,Beyond Empire Remastered,2011-03-13,"2,000,000 .. 5,000,000",59.99,923.0,722.0,172.0,"Massively Multiplayer, Platformer, Racing, Str..."
8,Lost Odyssey,2013-07-31,"100,000 .. 200,000",13.49,NaN,215.0,87.0,"Indie, Sports"
9,Silent Legends Deluxe,2020-04-24,"50,000 .. 100,000",1.79,1025.0,824.0,13920.0,Strategy


### Q2 — Games priced above $20

In [27]:
q2 = '''\nSELECT name, price, genres
FROM steam_games
WHERE price > 20
ORDER BY price DESC;\n'''

result = pd.read_sql_query(q2, conn)
display(result)

,name,price,genres
0,Iron Tactics Deluxe 2,59.99,"Sports, Puzzle"
1,Beyond Empire Remastered,59.99,"Massively Multiplayer, Platformer, Racing, Str..."
2,Last Tactics Online III,59.99,"Puzzle, Massively Multiplayer, Racing"
3,Baldur's Gate 3,59.99,"Indie, Horror, Adventure"
4,Shadow Rift Origins,59.99,"Adventure, Platformer"
...,...,...,...
160,SUBNAUTICA,22.49,None
161,Pixel Kingdom,22.49,Strategy
162,Mystic Horizon,22.49,"Rpg, Strategy, Free To Play"
163,RESIDENT EVIL 4,22.49,Free To Play


### Q3 — Top 10 games by total reviews

In [28]:
q3 = '''\nSELECT name, positive, negative,
       positive + negative AS total_reviews
FROM steam_games
WHERE positive IS NOT NULL AND negative IS NOT NULL
ORDER BY total_reviews DESC
LIMIT 10;\n'''

result = pd.read_sql_query(q3, conn)
display(result)

,name,positive,negative,total_reviews
0,Shadow Arena,33679.0,58.0,33737.0
1,Neon Empire Remastered,33412.0,158.0,33570.0
2,Lost Empire Online,32863.0,132.0,32995.0
3,Pixel Tactics Remastered,32785.0,77.0,32862.0
4,Shadow Empire,27018.0,138.0,27156.0
5,PIXEL SURVIVAL DELUXE,23709.0,97.0,23806.0
6,Iron Rift Deluxe,23474.0,30.0,23504.0
7,Frozen Rift Reborn,20686.0,58.0,20744.0
8,Shadow Odyssey Origins,19493.0,1082.0,20575.0
9,Lost Journey Remastered,20055.0,70.0,20125.0


In [29]:
result.to_csv(REPORT_DIR / 'sql_top_reviewed_games.csv', index=False)

### Q4 — Count all games

In [30]:
q4 = '''\nSELECT COUNT(*) AS total_games
FROM steam_games;\n'''

result = pd.read_sql_query(q4, conn)
display(result)

,total_games
0,491


### Q5 — Sum positive and negative reviews

In [31]:
q5 = '''\nSELECT SUM(positive) AS total_positive_reviews,
       SUM(negative) AS total_negative_reviews
FROM steam_games;\n'''

result = pd.read_sql_query(q5, conn)
display(result)

,total_positive_reviews,total_negative_reviews
0,1510009.0,138800.0


### Q6 — Average, minimum, and maximum price

In [32]:
q6 = '''\nSELECT AVG(price) AS average_price,
       MIN(price) AS minimum_price,
       MAX(price) AS maximum_price
FROM steam_games;\n'''

result = pd.read_sql_query(q6, conn)
display(result)

,average_price,minimum_price,maximum_price
0,17.934622,0.0,59.99


### Q7 — Count games by release year

In [33]:
q7 = '''\nSELECT strftime('%Y', release_date) AS release_year,
       COUNT(*) AS game_count
FROM steam_games
WHERE release_date IS NOT NULL
GROUP BY release_year
ORDER BY release_year;\n'''

result = pd.read_sql_query(q7, conn)
display(result)

,release_year,game_count
0,2005,22
1,2006,18
2,2007,27
3,2008,16
4,2009,18
5,2010,20
6,2011,23
7,2012,22
8,2013,27
9,2014,18


### Q8 — Satisfaction by price category (CASE WHEN)

In [34]:
q8 = '''\nSELECT
    CASE
        WHEN price = 0 THEN 'Free'
        WHEN price > 0 AND price <= 10 THEN 'Budget'
        WHEN price > 10 AND price <= 30 THEN 'Mid-range'
        WHEN price > 30 THEN 'Premium'
        ELSE 'Unknown'
    END AS price_category,
    AVG(100.0 * positive / NULLIF(positive + negative, 0)) AS average_positive_rate,
    COUNT(*) AS game_count
FROM steam_games
WHERE positive IS NOT NULL AND negative IS NOT NULL
GROUP BY price_category
ORDER BY average_positive_rate DESC;\n'''

result = pd.read_sql_query(q8, conn)
display(result)

,price_category,average_positive_rate,game_count
0,Unknown,91.274827,14
1,Free,88.370158,33
2,Premium,85.379716,93
3,Mid-range,83.145016,124
4,Budget,81.696772,202


### Q9 — Genre groups with at least 5 games (GROUP BY/HAVING)

In [35]:
q9 = '''\nSELECT genres, COUNT(*) AS game_count,
       AVG(price) AS average_price
FROM steam_games
WHERE genres IS NOT NULL
GROUP BY genres
HAVING COUNT(*) >= 5
ORDER BY game_count DESC;\n'''

result = pd.read_sql_query(q9, conn)
display(result)

,genres,game_count,average_price
0,Action,11,14.961000
1,Strategy,10,20.720000
2,Free To Play,9,10.677500
3,Casual,9,20.866250
4,Racing,7,13.156667
5,Puzzle,6,6.390000
6,Indie,6,15.658333
7,Horror,6,17.256667


### Q10 — Games with positive reviews above average (subquery)

In [36]:
q10 = '''\nSELECT name, positive
FROM steam_games
WHERE positive > (SELECT AVG(positive) FROM steam_games)
ORDER BY positive DESC;\n'''

result = pd.read_sql_query(q10, conn)
display(result)

,name,positive
0,Shadow Arena,33679.0
1,Neon Empire Remastered,33412.0
2,Lost Empire Online,32863.0
3,Pixel Tactics Remastered,32785.0
4,Shadow Empire,27018.0
...,...,...
114,Pixel Survival Deluxe,3426.0
115,Last Protocol Origins,3391.0
116,Cyber Horizon Remastered,3326.0
117,Lost Survival,3249.0


### Q11 — Positive rate above 80%, with 20+ reviews

In [37]:
q11 = '''\nSELECT name, positive, negative,
       ROUND(100.0 * positive / NULLIF(positive + negative, 0), 2) AS positive_rate
FROM steam_games
WHERE positive + negative >= 20
  AND 100.0 * positive / NULLIF(positive + negative, 0) > 80
ORDER BY positive_rate DESC;\n'''

result = pd.read_sql_query(q11, conn)
display(result)

,name,positive,negative,positive_rate
0,mystic rift remastered,8118.0,8.0,99.90
1,Iron Rift Deluxe,23474.0,30.0,99.87
2,Shadow Arena,33679.0,58.0,99.83
3,Mystic Odyssey Reborn,14570.0,29.0,99.80
4,Pixel Tactics Remastered,32785.0,77.0,99.77
...,...,...,...,...
323,Neon Survival Deluxe,1375.0,325.0,80.88
324,Warframe,910.0,217.0,80.75
325,Tiny Protocol Online,766.0,185.0,80.55
326,Stardew Valley,678.0,168.0,80.14


In [38]:
result.to_csv(REPORT_DIR / 'sql_high_satisfaction_games.csv', index=False)

### Q12 — Top 10 games by average playtime

In [39]:
q12 = '''\nSELECT name, average_playtime_forever, genres
FROM steam_games
WHERE average_playtime_forever IS NOT NULL
ORDER BY average_playtime_forever DESC
LIMIT 10;\n'''

result = pd.read_sql_query(q12, conn)
display(result)

,name,average_playtime_forever,genres
0,Forza Horizon 5,16140.0,"Massively Multiplayer, Indie"
1,Silent Legends Deluxe,13920.0,Strategy
2,Iron Dungeon Online,12960.0,"Platformer, Action"
3,Tiny Arena Reborn,9440.0,Horror
4,Forgotten Chronicles Reborn,7500.0,"Free To Play, Adventure, Sports"
5,Wild Journey Online,3041.0,"Horror, Platformer, Puzzle, Free To Play"
6,Cyber Dungeon Origins,2786.0,"Free To Play, Platformer"
7,Project Zomboid,2352.0,"Horror, Sports, Rpg, Indie"
8,BEYOND ARENA ONLINE,2311.0,"Massively Multiplayer, Simulation, Platformer,..."
9,Iron Chronicles Reborn,1905.0,"Platformer, Strategy"


### Q13 — Compare free and paid games (CASE WHEN)

In [40]:
q13 = '''\nSELECT
    CASE
        WHEN price = 0 THEN 'Free'
        WHEN price > 0 THEN 'Paid'
        ELSE 'Unknown'
    END AS game_type,
    COUNT(*) AS game_count,
    AVG(positive) AS average_positive_reviews,
    AVG(negative) AS average_negative_reviews,
    AVG(price) AS average_price
FROM steam_games
GROUP BY game_type;\n'''

result = pd.read_sql_query(q13, conn)
display(result)

,game_type,game_count,average_positive_reviews,average_negative_reviews,average_price
0,Free,34,3168.060606,157.352941,0.000000
1,Paid,442,3036.630233,303.487239,19.314208
2,Unknown,15,6647.466667,189.071429,NaN


### Q14 — Games with more negative than positive reviews

In [41]:
q14 = '''\nSELECT name, positive, negative,
       positive + negative AS total_reviews
FROM steam_games
WHERE negative > positive
ORDER BY total_reviews DESC;\n'''

result = pd.read_sql_query(q14, conn)
display(result)

,name,positive,negative,total_reviews
0,Pixel Frontier Online,1976.0,5118.0,7094.0
1,Tiny Arena Reborn,2177.0,2609.0,4786.0
2,Mystic Empire Remastered,1809.0,2957.0,4766.0
3,Forgotten Horizon Remastered,1922.0,2498.0,4420.0
4,Tiny Odyssey,1599.0,2137.0,3736.0
5,Forgotten Simulator Deluxe,1483.0,2125.0,3608.0
6,Silent Tactics Reborn,471.0,1749.0,2220.0
7,Forgotten Chronicles Online,669.0,1291.0,1960.0
8,Rust,631.0,1249.0,1880.0
9,Cyber Tactics Origins,611.0,1193.0,1804.0


### Q15 — Games priced above average price (subquery)

In [42]:
q15 = '''\nSELECT name, price
FROM steam_games
WHERE price > (SELECT AVG(price) FROM steam_games)
ORDER BY price DESC;\n'''

result = pd.read_sql_query(q15, conn)
display(result)

,name,price
0,Iron Tactics Deluxe 2,59.99
1,Beyond Empire Remastered,59.99
2,Last Tactics Online III,59.99
3,Baldur's Gate 3,59.99
4,Shadow Rift Origins,59.99
...,...,...
187,Iron Legends Origins,17.99
188,Last Protocol Deluxe,17.99
189,Wild Journey Online,17.99
190,Forgotten Odyssey Origins,17.99


### Q16 — Missing values in key columns (CASE WHEN)

In [43]:
q16 = '''\nSELECT COUNT(*) AS total_games,
       SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END) AS missing_price,
       SUM(CASE WHEN positive IS NULL THEN 1 ELSE 0 END) AS missing_positive,
       SUM(CASE WHEN negative IS NULL THEN 1 ELSE 0 END) AS missing_negative,
       SUM(CASE WHEN average_playtime_forever IS NULL THEN 1 ELSE 0 END) AS missing_playtime
FROM steam_games;\n'''

result = pd.read_sql_query(q15, conn)
display(result)

,name,price
0,Iron Tactics Deluxe 2,59.99
1,Beyond Empire Remastered,59.99
2,Last Tactics Online III,59.99
3,Baldur's Gate 3,59.99
4,Shadow Rift Origins,59.99
...,...,...
187,Iron Legends Origins,17.99
188,Last Protocol Deluxe,17.99
189,Wild Journey Online,17.99
190,Forgotten Odyssey Origins,17.99


## Export all SQL statements to a `.sql` file

In [44]:
sql_text = """-- STEAM INSIGHT | SQL Business Analysis
-- Table: steam_games

"""

# Get SQL query strings from your notebook variables
query_names = [
    "q1", "q2", "q3", "q4", "q5", "q6",
    "q7", "q8", "q9", "q10", "q11", "q12",
    "q13", "q14", "q15", "q16"
]

for i, query_name in enumerate(query_names, start=1):
    if query_name in globals():
        sql_text += f"-- Query {i}\n"
        sql_text += globals()[query_name].strip() + "\n\n"

sql_file = SQL_DIR / "steam_insight_queries.sql"
sql_file.write_text(sql_text, encoding="utf-8")

print("SQL file saved:", sql_file)

SQL file saved: ..\sql\steam_insight_queries.sql


In [45]:
conn.close()